In [30]:
import yfinance as yf
import pytz
import pandas as pd

In [36]:
import os, json
# project_root = os.path.dirname(os.path.abspath(__file__))
# portfolio_path=os.path.join(project_root, "portfolio.json")
portfolio_path = "D:\\Dev\\pfa-backend-fastapi\\app\\portfolio.json"
print(portfolio_path)
def load_tickers_from_portfolio():
    with open(portfolio_path, "r") as f:
        data = json.load(f)
    tickers = []
    for i in range(0,len(data["equities"])):
        tickers.append(data["equities"][i]["ticker"])
    return tickers

D:\Dev\pfa-backend-fastapi\app\portfolio.json


In [39]:
def get_stock_OHLCV_data(tickers, interval, period):
    result = {}
    # set timezone
    eastern_tz = pytz.timezone("US/Eastern")
    for ticker in tickers:
        # get intraday OHLCV info for each ticker
        df = yf.download(ticker, interval=interval, period=period, progress=False)
        # convert the recieved timestamp to EST time zone
        df.index = df.index.tz_convert(eastern_tz)
        df = df.between_time("09:30", "17:30")
        # Drop last row if it's not a full interval
        if df.index[-1].time().strftime('%H:%M') != "17:30":
            df = df.iloc[:-1]
        print(df)
        result[ticker] = df
    return result

In [33]:
# --- Format as compact JSON (summary version for LLMs) ---
def format_summary_json(data):
    summary = {}
    for ticker, df in data.items():
        latest = df.iloc[-1]
        first = df.iloc[0]
        open_price = float(first['Open'].iloc[0])
        close_price = float(latest['Close'].iloc[0])
        change_percent = round(((close_price - open_price) / open_price) * 100, 2)

        summary[ticker] = {
            "open": round(open_price, 2),
            "close": round(close_price, 2),
            "change_percent": change_percent,
            "volume_sum": int(df['Volume'].sum().iloc[0]),
            "intervals": len(df),
        }
    return summary


In [34]:
# --- Format as time-series table (Excel-like) ---
def format_time_series_table(data):
    table = {}
    for ticker, df in data.items():
        table[ticker] = []
        for idx, row in df.iterrows():
            table[ticker].append({
                "time": idx.strftime('%Y-%m-%d %H:%M'),
                "open": float(row['Open'].iloc[0]),
                "high": float(row['High'].iloc[0]),
                "low": float(row['Low'].iloc[0]),
                "close": float(row['Close'].iloc[0]),
                "volume": int(row['Volume'].iloc[0])
            })
    return table

In [41]:
tickers = load_tickers_from_portfolio()
print(tickers)
stock_data = get_stock_OHLCV_data(tickers, "30m", "1d")
stock_data

['AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOG', 'TSLA', 'META', 'BRK-A', 'JPM', 'COST', 'CRWD', 'SHOP', 'DKNG', 'NET', 'RKLB', 'CELH', 'AXON', 'JNJ', 'PG', 'KO', 'VZ', 'O', 'T', 'PEP', 'ABBV']
Price                           Close        High         Low        Open  \
Ticker                           AAPL        AAPL        AAPL        AAPL   
Datetime                                                                    
2025-05-08 09:30:00-04:00  196.149994  198.130005  196.082108  197.740005   
2025-05-08 10:00:00-04:00  195.820007  196.309998  194.679596  196.149994   
2025-05-08 10:30:00-04:00  196.619995  197.249695  195.514999  195.830002   
2025-05-08 11:00:00-04:00  197.539993  197.830002  196.419998  196.580002   
2025-05-08 11:30:00-04:00  199.339996  199.750000  197.309998  197.529999   
2025-05-08 12:00:00-04:00  199.240005  200.049805  199.229996  199.339996   
2025-05-08 12:30:00-04:00  199.695007  199.990005  199.010101  199.235703   
2025-05-08 13:00:00-04:00  199.139999  199.8

{'AAPL': Price                           Close        High         Low        Open  \
 Ticker                           AAPL        AAPL        AAPL        AAPL   
 Datetime                                                                    
 2025-05-08 09:30:00-04:00  196.149994  198.130005  196.082108  197.740005   
 2025-05-08 10:00:00-04:00  195.820007  196.309998  194.679596  196.149994   
 2025-05-08 10:30:00-04:00  196.619995  197.249695  195.514999  195.830002   
 2025-05-08 11:00:00-04:00  197.539993  197.830002  196.419998  196.580002   
 2025-05-08 11:30:00-04:00  199.339996  199.750000  197.309998  197.529999   
 2025-05-08 12:00:00-04:00  199.240005  200.049805  199.229996  199.339996   
 2025-05-08 12:30:00-04:00  199.695007  199.990005  199.010101  199.235703   
 2025-05-08 13:00:00-04:00  199.139999  199.850006  199.050003  199.720001   
 2025-05-08 13:30:00-04:00  198.321304  199.212006  198.303299  199.139999   
 2025-05-08 14:00:00-04:00  198.639999  198.664993  198.

In [44]:
summary_output = format_summary_json(stock_data)
summary_output

{'AAPL': {'open': 197.74,
  'close': 199.56,
  'change_percent': 0.92,
  'volume_sum': 32893554,
  'intervals': 12},
 'MSFT': {'open': 437.93,
  'close': 442.12,
  'change_percent': 0.96,
  'volume_sum': 15323972,
  'intervals': 12},
 'NVDA': {'open': 118.48,
  'close': 118.54,
  'change_percent': 0.05,
  'volume_sum': 162259579,
  'intervals': 12},
 'AMZN': {'open': 191.48,
  'close': 193.92,
  'change_percent': 1.28,
  'volume_sum': 28205299,
  'intervals': 12},
 'GOOG': {'open': 155.85,
  'close': 157.0,
  'change_percent': 0.74,
  'volume_sum': 31282063,
  'intervals': 12},
 'TSLA': {'open': 279.63,
  'close': 289.02,
  'change_percent': 3.36,
  'volume_sum': 83161079,
  'intervals': 12},
 'META': {'open': 606.28,
  'close': 604.68,
  'change_percent': -0.27,
  'volume_sum': 11078724,
  'intervals': 12},
 'BRK-A': {'open': 780091.56,
  'close': 773999.5,
  'change_percent': -0.78,
  'volume_sum': 363,
  'intervals': 12},
 'JPM': {'open': 251.57,
  'close': 255.57,
  'change_percent

In [46]:
table_output = format_time_series_table(stock_data)
table_output

{'AAPL': [{'time': '2025-05-08 09:30',
   'open': 197.74000549316406,
   'high': 198.1300048828125,
   'low': 196.0821075439453,
   'close': 196.14999389648438,
   'volume': 5831936},
  {'time': '2025-05-08 10:00',
   'open': 196.14999389648438,
   'high': 196.30999755859375,
   'low': 194.67959594726562,
   'close': 195.82000732421875,
   'volume': 4617763},
  {'time': '2025-05-08 10:30',
   'open': 195.8300018310547,
   'high': 197.24969482421875,
   'low': 195.51499938964844,
   'close': 196.6199951171875,
   'volume': 3058573},
  {'time': '2025-05-08 11:00',
   'open': 196.5800018310547,
   'high': 197.8300018310547,
   'low': 196.4199981689453,
   'close': 197.5399932861328,
   'volume': 2570973},
  {'time': '2025-05-08 11:30',
   'open': 197.52999877929688,
   'high': 199.75,
   'low': 197.30999755859375,
   'close': 199.33999633789062,
   'volume': 4082906},
  {'time': '2025-05-08 12:00',
   'open': 199.33999633789062,
   'high': 200.0498046875,
   'low': 199.22999572753906,
   

In [47]:
print("\n--- COMPACT SUMMARY FORMAT ---")
for ticker, summary in summary_output.items():
    print(f"{ticker}: {summary}")

print("\n--- FULL TIME-SERIES TABLE FORMAT ---")
for ticker, rows in table_output.items():
    print(f"\n{ticker}:")
    for row in rows:
        print(row)


--- COMPACT SUMMARY FORMAT ---
AAPL: {'open': 197.74, 'close': 199.56, 'change_percent': 0.92, 'volume_sum': 32893554, 'intervals': 12}
MSFT: {'open': 437.93, 'close': 442.12, 'change_percent': 0.96, 'volume_sum': 15323972, 'intervals': 12}
NVDA: {'open': 118.48, 'close': 118.54, 'change_percent': 0.05, 'volume_sum': 162259579, 'intervals': 12}
AMZN: {'open': 191.48, 'close': 193.92, 'change_percent': 1.28, 'volume_sum': 28205299, 'intervals': 12}
GOOG: {'open': 155.85, 'close': 157.0, 'change_percent': 0.74, 'volume_sum': 31282063, 'intervals': 12}
TSLA: {'open': 279.63, 'close': 289.02, 'change_percent': 3.36, 'volume_sum': 83161079, 'intervals': 12}
META: {'open': 606.28, 'close': 604.68, 'change_percent': -0.27, 'volume_sum': 11078724, 'intervals': 12}
BRK-A: {'open': 780091.56, 'close': 773999.5, 'change_percent': -0.78, 'volume_sum': 363, 'intervals': 12}
JPM: {'open': 251.57, 'close': 255.57, 'change_percent': 1.59, 'volume_sum': 5227653, 'intervals': 12}
COST: {'open': 1013.5,

D:\Dev\pfa-backend-fastapi\app\portfolio.json


['AAPL',
 'MSFT',
 'NVDA',
 'AMZN',
 'GOOG',
 'TSLA',
 'META',
 'BRK.B',
 'JPM',
 'COST',
 'CRWD',
 'SHOP',
 'DKNG',
 'NET',
 'RKLB',
 'CELH',
 'AXON',
 'JNJ',
 'PG',
 'KO',
 'VZ',
 'O',
 'T',
 'PEP',
 'ABBV']

['AAPL', 'AAPL']